# CS 195: Natural Language Processing
## LSTM and Encoder-Decoder Architectures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F6_1_LSTMEncoderDecoder.ipynb)

## Reference

SLP: RNNs and LSTMs, Chapter 13 of Speech and Language Processing by Daniel Jurafsky & James H. Martin https://web.stanford.edu/~jurafsky/slp3/13.pdf


In [3]:
#import sys
#!{sys.executable} -m pip install torch datasets transformers

## Last time: RNN Language Model

We used recurrent neural networks for *language modeling* - predicting the next word.

<div>
    <img src="images/RNN_languagemodeling.png" width=700>
</div>


image source: SLP Fig. 9.6, https://web.stanford.edu/~jurafsky/slp3/9.pdf

## Workflow from last time

In [5]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer
from datasets import load_dataset
from torch.utils.data import TensorDataset, DataLoader

random.seed(42)
torch.manual_seed(42)

class TinyRNNLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()

        # Look up a learned vector for each token id
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)

        # Read the sequence left-to-right, updating a hidden state at each step
        # batch_first=True shapes the data like this [batch_size, sequence_length, embedding_dim]
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

        # Turn the final hidden state into scores for every token in the vocabulary
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        # input_ids shape: [batch_size, sequence_length]
        emb = self.embedding(input_ids)

        # emb shape: [batch_size, sequence_length, embedding_dim]
        # hidden shape: [1, batch_size, hidden_dim]
        rnn_out, hidden = self.rnn(emb)

        # Remove the extra first dimension since we only have one RNN layer
        last_hidden = hidden.squeeze(0)

        # logits shape: [batch_size, vocab_size]
        logits = self.output(last_hidden)
        return logits


def make_lm_examples(sentences, tokenizer, window_size=4):
    X = []
    y = []

    for sentence in sentences:
        ids = tokenizer(sentence, add_special_tokens=False)["input_ids"]
        for i in range(1, len(ids)):
            # Grab the recent context before the target token
            prefix = ids[max(0, i - window_size):i]
            # Add [PAD] tokens on the left if the context is shorter than the window
            prefix = [pad_id] * (window_size - len(prefix)) + prefix
            X.append(prefix)
            y.append(ids[i])

    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

def generate_text(start_text, model, tokenizer, num_tokens=8, window_size=4):

    # we're not training, so set the model to eval mode
    model.eval()
    generated_ids = tokenizer(start_text, add_special_tokens=False)["input_ids"]

    for _ in range(num_tokens):
        # Grab the recent context before the target token
        prefix = generated_ids[-window_size:]
        # Add [PAD] tokens on the left if the context is shorter than the window
        prefix = [pad_id] * (window_size - len(prefix)) + prefix
        x = torch.tensor([prefix], dtype=torch.long)
        x = x.to(device)

        with torch.no_grad():
            logits = model(x)
            next_id = logits.argmax(dim=1).item()

        generated_ids.append(next_id)

    tokens = tokenizer.convert_ids_to_tokens([i for i in generated_ids if i != pad_id and i != unk_id])
    return tokens

def train_model(loader,loss_fn,optimizer,num_epochs=10):
    # set the model to be in training mode
    model.train()
    
    
    for epoch in range(num_epochs):
        # We'll accumulate statistics across all batches so we can report one loss/accuracy per epoch
        total_loss = 0.0
        total_correct = 0
        total_examples = 0
    
        # New idea: instead of pushing all training examples through the model at once,
        # the DataLoader gives us one small batch at a time
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
    
            # This part is the same as before, just on one batch instead of all of X_train / y_train
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
            # Keep track of how this batch did so we can average over the whole epoch
            batch_size = X_batch.size(0) # X_batch has shape [batch_size, sequence_length], so this gives us the batch_size
            total_loss += loss.item() * batch_size
            predictions = logits.argmax(dim=1)
            total_correct += (predictions == y_batch).sum().item()
            total_examples += batch_size
    
        # Turn the accumulated totals into epoch-level averages
        avg_loss = total_loss / total_examples
        acc = total_correct / total_examples
        print(f"epoch {epoch+1}, loss={avg_loss:.4f}, acc={acc:.4f}")

dataset = load_dataset("sarnab/Shakespeare_Corpus")
sentences = dataset['train']['text']

print("Example sentence:",sentences[1000])

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
unk_id = tokenizer.unk_token_id

vocab_size = tokenizer.vocab_size
print("vocab size:", vocab_size)

window_size = 4
pad_id = tokenizer.pad_token_id



X_train, y_train = make_lm_examples(sentences, tokenizer, window_size=window_size)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("example input ids:", X_train[0])
print("example target id:", y_train[0].item())


device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")


dataset = TensorDataset(X_train, y_train)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

model = TinyRNNLM(vocab_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
model.to(device)
train_model(loader,loss_fn,optimizer,num_epochs=15)



Example sentence: First Senator: Do you hear how we are shent for keeping your greatness back?


Token indices sequence length is longer than the specified maximum sequence length for this model (594 > 512). Running this sequence through the model will result in indexing errors


vocab size: 30522
X_train shape: torch.Size([281497, 4])
y_train shape: torch.Size([281497])
example input ids: tensor([   0,    0,    0, 2034])
example target id: 6926
Using device: mps
epoch 1, loss=5.8246, acc=0.1426
epoch 2, loss=5.0195, acc=0.1940
epoch 3, loss=4.7823, acc=0.2044
epoch 4, loss=4.6397, acc=0.2109
epoch 5, loss=4.5401, acc=0.2166
epoch 6, loss=4.4659, acc=0.2210
epoch 7, loss=4.4048, acc=0.2255
epoch 8, loss=4.3597, acc=0.2291
epoch 9, loss=4.3235, acc=0.2315
epoch 10, loss=4.2951, acc=0.2340
epoch 11, loss=4.2699, acc=0.2360
epoch 12, loss=4.2510, acc=0.2378
epoch 13, loss=4.2422, acc=0.2382
epoch 14, loss=4.2270, acc=0.2403
epoch 15, loss=4.2181, acc=0.2404


In [6]:
print(generate_text("romeo", model, tokenizer))
print(generate_text("thou", model, tokenizer))
print(generate_text("the king", model, tokenizer))

['romeo', ':', 'what', 'is', 'the', 'jay', 'of', 'your', 'grace']
['thou', '##eni', '##us', ':', 'i', "'", 'll', 'not', 'pass']
['the', 'king', 'vincent', '##io', ',', 'and', 'i', "'", 'll', 'be']


## What Are the Limitations of This Model?

This example shows the core causal LM idea, but plain RNNs have important limits:
- it is hard for them to remember information from far back in the sequence
- they process tokens one step at a time, so computation is inherently sequential
- training can become unstable on longer sequences

This is where LSTMs and GRUs enter the story: they are recurrent models designed to preserve information better across time.
Later, attention will give a different solution to the same general problem.

## LSTM and GRU

One of the main problems with plain RNNs is that it is hard for training signals and useful information to survive across many time steps.
This is related to the **vanishing gradient** problem.

<div>
    <img src="images/vanishing_gradient.png" width=520>
</div>

Historically, this led to gated recurrent models like **LSTMs** (Long Short-Term Memory) and **GRUs** (Gated Recurrent Unit).
They add mechanisms for deciding what information to keep, update, or forget. 
* The $c$ in these diagrams is an explicit context vector that can be passed down the line
* **forget gate** - it learns what to delete from the context and applied a mask element-by-element
* **add gate** - learn what to add in from the current context
<div>
    <img src="images/RNN-vs-LSTM-vs-GRU.png" width=650>
</div>

<div>
    <img src="images/LSTM_node.png" width=650>
</div>

I don't want to spend a whole day experimenting with these, but you can feel free to peform some experiments with them as an *Extended Implementation Creative Synthesis* project.

Things to keep in mind about them, though
- plain RNNs introduced sequence-aware modeling
- LSTMs and GRUs improved long-term memory

**You can use an LSTM or GRU instead anywhere you'd use the basic RNN unit**

image credits: https://web.stanford.edu/~jurafsky/slp3

## RNN for Sequence Classification

We could also use the last hidden state an RNN as an input to a regular feed-forward network and do classification of a whole sequence.

<div>
    <img src="images/RNN_classification.png" width=700>
</div>


image source: SLP Fig. 13.8, https://web.stanford.edu/~jurafsky/slp3/13.pdf

## RNN Sequence Labeling

RNNs are also good for **sequence labeling** when the output is a squence corresponding 1:1 with the input words, like part-of-speech tagging.

<div>
    <img src="images/RNN_sequence_labeling.png" width=700>
</div>


image source: SLP Fig. 13.7, https://web.stanford.edu/~jurafsky/slp3/13.pdf

### Discussion Question

What sequence-to-sequence NLP tasks can you think of where the input and target sequences don't match up word-for word?

## Encoder-Decoder Architecture

**Encoder RNN:** Takes input sequences, produces a context vector

**Context Vector:** Contains essence of the input sequence

**Decoder RNN:** Takes context vector as input, generates an output sequence

<div>
    <img src="images/encoder-decoder.png" width=700>
</div>


image source: SLP Fig. 13.16, https://web.stanford.edu/~jurafsky/slp3/13.pdf

## Encoder-Decoder usage

<div>
    <img src="images/encoder-decoder_detail.png" width=800>
</div>


image source: SLP Fig. 13.18, https://web.stanford.edu/~jurafsky/slp3/13.pdf

## Discussion Questions

Think of a sequence-to-sequence learning task. Write down one that you have in mind.

Think about *training* this model for your task. 
* What would you *input* as the $x$s in this model?
* What would you *input* as the $y$s in this model?

Think about using this model for *inference* with your task.
* What would you *input* as the $x$s?
* What would you *input* as the $y$s?

## A Synthetic Seq2Seq Task

Instead of jumping straight into a messy real-world task, let's use a small **synthetic** sequence-to-sequence problem.

**Task:** given a short sequence of tokens, predict the same sequence in reverse order.

Examples:
- input: `red dog runs`
- output: `runs dog red`

## Build a Tiny Vocabulary and Synthetic Dataset

We will make short random sequences from a small vocabulary.

We'll also add special tokens:
- `<PAD>` for padding
- `<SOS>` for the start of decoder generation
- `<EOS>` for the end of the output sequence


In [4]:
base_tokens = [
    'red', 'blue', 'green', 'yellow',
    'dog', 'cat', 'bird', 'fish',
    'runs', 'jumps', 'swims', 'sleeps'
]

special_tokens = ['<PAD>', '<SOS>', '<EOS>']
vocab = special_tokens + base_tokens

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

pad_id = word_to_id['<PAD>']
sos_id = word_to_id['<SOS>']
eos_id = word_to_id['<EOS>']
vocab_size = len(vocab)

print('vocab size:', vocab_size)
print(word_to_id)


def make_reverse_examples(num_examples=1000, min_len=3, max_len=5):
    inputs = []
    targets = []

    for _ in range(num_examples):
        seq_len = random.randint(min_len, max_len)
        tokens = random.sample(base_tokens, seq_len)
        reversed_tokens = list(reversed(tokens))
        inputs.append(tokens)
        targets.append(reversed_tokens)

    return inputs, targets


input_sequences, target_sequences = make_reverse_examples()
print(input_sequences[:3])
print(target_sequences[:3])


vocab size: 15
{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, 'red': 3, 'blue': 4, 'green': 5, 'yellow': 6, 'dog': 7, 'cat': 8, 'bird': 9, 'fish': 10, 'runs': 11, 'jumps': 12, 'swims': 13, 'sleeps': 14}
[['blue', 'red', 'dog', 'yellow', 'runs'], ['sleeps', 'blue', 'runs'], ['jumps', 'bird', 'red']]
[['runs', 'yellow', 'dog', 'red', 'blue'], ['runs', 'blue', 'sleeps'], ['red', 'bird', 'jumps']]


## Convert the Token Sequences to Training Tensors

For encoder-decoder training, we need three versions of the output sequence:
- the **encoder input**: the source sequence
- the **decoder input**: starts with `<SOS>` and then the target tokens
- the **decoder target**: the target tokens followed by `<EOS>`

This lets us use **teacher forcing** during training: at each decoder step, we give the decoder the correct previous token rather than its own prediction.


In [5]:
max_input_len = max(len(seq) for seq in input_sequences)
max_target_len = max(len(seq) for seq in target_sequences) + 1  # room for <EOS>


def encode_input(seq):
    ids = [word_to_id[word] for word in seq]
    ids = ids + [pad_id] * (max_input_len - len(ids))
    return ids


def encode_decoder_input(seq):
    ids = [sos_id] + [word_to_id[word] for word in seq]
    ids = ids + [pad_id] * (max_target_len - len(ids))
    return ids


def encode_decoder_target(seq):
    ids = [word_to_id[word] for word in seq] + [eos_id]
    ids = ids + [pad_id] * (max_target_len - len(ids))
    return ids


encoder_inputs = torch.tensor([encode_input(seq) for seq in input_sequences], dtype=torch.long)
decoder_inputs = torch.tensor([encode_decoder_input(seq) for seq in target_sequences], dtype=torch.long)
decoder_targets = torch.tensor([encode_decoder_target(seq) for seq in target_sequences], dtype=torch.long)

print('encoder_inputs shape:', encoder_inputs.shape)
print('decoder_inputs shape:', decoder_inputs.shape)
print('decoder_targets shape:', decoder_targets.shape)
print('example encoder input:', encoder_inputs[0])
print('example decoder input:', decoder_inputs[0])
print('example decoder target:', decoder_targets[0])


encoder_inputs shape: torch.Size([1000, 5])
decoder_inputs shape: torch.Size([1000, 6])
decoder_targets shape: torch.Size([1000, 6])
example encoder input: tensor([ 4,  3,  7,  6, 11])
example decoder input: tensor([ 1, 11,  6,  7,  3,  4])
example decoder target: tensor([11,  6,  7,  3,  4,  2])


In [6]:
example_i = 0
print('source tokens:', input_sequences[example_i])
print('target tokens:', target_sequences[example_i])
print('decoder input tokens:', [id_to_word[i.item()] for i in decoder_inputs[example_i]])
print('decoder target tokens:', [id_to_word[i.item()] for i in decoder_targets[example_i]])


source tokens: ['blue', 'red', 'dog', 'yellow', 'runs']
target tokens: ['runs', 'yellow', 'dog', 'red', 'blue']
decoder input tokens: ['<SOS>', 'runs', 'yellow', 'dog', 'red', 'blue']
decoder target tokens: ['runs', 'yellow', 'dog', 'red', 'blue', '<EOS>']


## Make Train/Test Splits and DataLoaders

We'll batch the encoder inputs, decoder inputs, and decoder targets together.


In [14]:
num_examples = len(encoder_inputs)
indices = list(range(num_examples))
random.shuffle(indices)

split = int(0.8 * num_examples)
train_idx = indices[:split]
test_idx = indices[split:]

train_dataset = TensorDataset(
    encoder_inputs[train_idx],
    decoder_inputs[train_idx],
    decoder_targets[train_idx],
)

# We're not actually going to validate with this dataset today
test_dataset = TensorDataset(
    encoder_inputs[test_idx],
    decoder_inputs[test_idx],
    decoder_targets[test_idx],
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

print('train examples:', len(train_dataset))
print('test examples:', len(test_dataset))


train examples: 800
test examples: 200


## Encoder and Decoder in PyTorch

We'll test out a **GRU** instead of a plain **RNN**:
- it is still a recurrent model

The encoder reads the input sequence and returns a final hidden state.
The decoder starts from that hidden state and generates the output sequence one step at a time.


In [8]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        outputs, hidden = self.gru(emb)
        return outputs, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, decoder_input_ids, hidden):
        emb = self.embedding(decoder_input_ids)
        outputs, hidden = self.gru(emb, hidden)
        logits = self.output(outputs)
        return logits, hidden


class Seq2SeqGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.encoder = Encoder(vocab_size, embedding_dim, hidden_dim)
        self.decoder = Decoder(vocab_size, embedding_dim, hidden_dim)

    def forward(self, encoder_input_ids, decoder_input_ids):
        encoder_outputs, hidden = self.encoder(encoder_input_ids)
        logits, hidden = self.decoder(decoder_input_ids, hidden)
        return logits


## Train with Teacher Forcing

During training, we already know the correct output sequence.
So instead of feeding the decoder its own previous prediction, we feed it the correct previous token from `decoder_inputs`.

This is called **teacher forcing**.
It usually makes training much easier.


In [9]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

model = Seq2SeqGRU(vocab_size).to(device)
loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(30):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for enc_batch, dec_in_batch, dec_tgt_batch in train_loader:
        enc_batch = enc_batch.to(device)
        dec_in_batch = dec_in_batch.to(device)
        dec_tgt_batch = dec_tgt_batch.to(device) # [batch_size, target_len]

        logits = model(enc_batch, dec_in_batch)  # [batch_size, target_len, vocab_size]

        # logits are a 3D tensor now, so we need to reshape it for CrossEntropyLoss which wants a 2D tensor
        # reshape with -1 keeps the last dimension as vocab_size and flattens the rest
        logits_2D = logits.reshape(-1, vocab_size) # [batch_size * target_len, vocab_size]
        dec_tgt_batch_1D = dec_tgt_batch.reshape(-1)

        loss = loss_fn(logits_2D, dec_tgt_batch_1D )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        non_pad_tokens = (dec_tgt_batch != pad_id).sum().item() # count the number of non-pad tokens
        total_loss += loss.item() * non_pad_tokens
        total_tokens += non_pad_tokens

    avg_loss = total_loss / total_tokens
    if (epoch + 1) % 5 == 0:
        print(f'epoch {epoch+1}, avg token loss={avg_loss:.4f}')


Using device: mps
epoch 5, avg token loss=0.2521
epoch 10, avg token loss=0.0436
epoch 15, avg token loss=0.0451
epoch 20, avg token loss=0.0339
epoch 25, avg token loss=0.0310
epoch 30, avg token loss=0.0082


## Discussion Question

Think about the inputs to this model

`logits = model(enc_batch, dec_in_batch)`

Write down an example of what the *inputs* look like. Look at the data we prepared up above.
* What data is in `enc_batch`?
* What data is in `dec_in_batch`?

How is *teacher forcing* happening here?


## Greedy Inference

At inference time, we do **not** know the correct output sequence.
So we:
- encode the input once
- start the decoder with `<SOS>`
- predict one token
- feed that predicted token back into the decoder
- stop when we produce `<EOS>` or hit a maximum length


In [12]:
def decode_greedy(input_tokens, model, max_steps=max_target_len):
    # some layers behave differently during training/inference like Dropout layers
    # this is probably not doing anything in this example
    model.eval()

    encoder_ids = encode_input(input_tokens)
    encoder_tensor = torch.tensor([encoder_ids], dtype=torch.long).to(device)

    # .no_grad() tells PyTorch not to track gradients
    with torch.no_grad():

        # we only use the encoder part of the model!
        encoder_outputs, hidden = model.encoder(encoder_tensor)

        # Now to build the decoder input - we don't have a teacher forcing example to use
        # so let's start with the start-of-sequence token
        start_token_id = sos_id
        
        # Make a batch with one sequence containing just that one token
        # Shape: [batch_size=1, sequence_length=1]
        decoder_input = [[start_token_id]]
        
        decoder_input = torch.tensor(decoder_input, dtype=torch.long)
        
        decoder_input = decoder_input.to(device)
        
        generated_ids = []

        for _ in range(max_steps):
            logits, hidden = model.decoder(decoder_input, hidden)
            
            # Shape goes from [batch_size, sequence_length, vocab_size]
            # to [batch_size, vocab_size] and we only want the one from the last time step
            last_step_logits = logits[:, -1, :]
            
            # Pick the vocabulary item with the highest score for each example in the batch
            # Shape: [batch_size]
            predicted_token_ids = last_step_logits.argmax(dim=1)
            
            # Since we only have one example in the batch, pull out that one token id as a Python number
            next_id = predicted_token_ids.item()

            if next_id == eos_id:
                break

            # prepare the just-predicted token as the decoder input on the next step
            generated_ids.append(next_id)
            decoder_input = torch.tensor([[next_id]], dtype=torch.long).to(device)

    return [id_to_word[i] for i in generated_ids]


## Try the Model

The prediction should look like the source sequence in reverse order.


In [13]:
for i in range(5):
    source = input_sequences[test_idx[i]]
    target = target_sequences[test_idx[i]]
    prediction = decode_greedy(source, model)
    print('source:    ', source)
    print('target:    ', target)
    print('predicted: ', prediction)
    print()


source:     ['dog', 'red', 'yellow', 'sleeps']
target:     ['sleeps', 'yellow', 'red', 'dog']
predicted:  ['sleeps', 'yellow', 'red', 'dog']

source:     ['jumps', 'sleeps', 'red', 'blue']
target:     ['blue', 'red', 'sleeps', 'jumps']
predicted:  ['blue', 'red', 'sleeps', 'jumps']

source:     ['green', 'swims', 'jumps', 'bird']
target:     ['bird', 'jumps', 'swims', 'green']
predicted:  ['bird', 'jumps', 'swims', 'green']

source:     ['red', 'blue', 'sleeps', 'jumps', 'bird']
target:     ['bird', 'jumps', 'sleeps', 'blue', 'red']
predicted:  ['bird', 'jumps', 'sleeps', 'blue', 'red']

source:     ['blue', 'bird', 'runs']
target:     ['runs', 'bird', 'blue']
predicted:  ['runs', 'bird', 'blue']



## Discussion Question

For inference, we still predict one word at a time with the decoder

`logits, hidden = model.decoder(decoder_input, hidden)`

Where did `hidden` come from here? Is it the same value every time we predict a new word, or does it update over time?


## What This Helps Us See About Encoder-Decoder Models

This toy task is much simpler than translation or summarization, but it lets us see the core mechanics clearly:
- the **encoder** reads the input sequence into a hidden representation
- the **decoder** uses that representation to generate an output sequence
- during training we often use **teacher forcing**
- during inference we have to feed predictions back in one step at a time

This is the key bridge from language modeling to sequence-to-sequence learning.


## Applied Exploration

Start from this synthetic reverse-sequence task and try one or more of these extensions:

1. Increase the sequence length and see when the model starts to struggle.
2. Replace the `GRU` with an `LSTM` and compare the results.
3. Replace the reverse task with another synthetic task such as copying, sorting by a simple rule, or mapping digits to words.

Write up what you changed, why you changed it, and how the model's behavior changed.

## Extended Implementation Idea

There's a bit of work that needs to be done to train on a real dataset. Try a HuggingFace sequence-to-sequence dataset like
https://huggingface.co/datasets/abisee/cnn_dailymail

